# chunking

In [1]:
import json
with open(r"..\data\processed/chunks.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

split_records = [r for r in records if r["was_split"]]
print("Total chunks:", len(records))
print("Chunks from split records:", len(split_records))
print("Max chunk_index seen:", max(r["chunk_index"] for r in records))

Total chunks: 9272
Chunks from split records: 6974
Max chunk_index seen: 8


In [5]:
from pathlib import Path
print(Path.cwd().parent)

d:\Programing\Github\Persian-Medical-RAG-Chatbot


## Evaluation chunk_size and chunk_overlap

In [1]:
"""
Evaluate which chunk_size/overlap setting retrieves better results.
"""
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")

from pathlib import Path

import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

PROJECT_ROOT = Path.cwd().parent
CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_dataset.csv"

LONG_RECORD_THRESHOLD = 1000
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
TOP_K = 5

# The configurations to compare - add or edit freely
CONFIGS = [
    {"name": "1000/200 (original)", "chunk_size": 1000, "chunk_overlap": 200},
    {"name": "400/60 (proposed middle ground)", "chunk_size": 400, "chunk_overlap": 60},
    {"name": "250/30 (your latest change)", "chunk_size": 250, "chunk_overlap": 30},
]


def build_test_set(df: pd.DataFrame) -> list[dict]:
    """Build (query, expected_drug) pairs from the long records only.

    IMPORTANT: the query must NOT be a verbatim substring of the source
    document, or retrieval becomes trivial (near 100% for every config,
    regardless of chunk size - this was the bug in the first version of
    this script). Instead, we pull a short excerpt from deep inside the
    RESPONSE text (offset ~60% into it). This tests whether information
    located later in a long record survives being split into a separate
    chunk under different chunk_size settings - which is exactly the
    scenario where chunk_size actually matters.
    """
    long_df = df[df["retrieval_text"].str.len() >= LONG_RECORD_THRESHOLD]
    test_set = []
    for _, row in long_df.iterrows():
        response = row["response_clean"]
        if not isinstance(response, str) or len(response) < 60:
            continue
        # Take a ~40-char excerpt starting 60% into the response text
        start = int(len(response) * 0.6)
        excerpt = response[start:start + 40].strip()
        if len(excerpt) < 20:
            continue
        test_set.append({"query": excerpt, "expected_drug": row["drug_name"]})
    return test_set


def build_vectorstore_for_config(long_df: pd.DataFrame, embedding, chunk_size: int, chunk_overlap: int) -> FAISS:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " .", ". ", ".", ":", "؟", "?", "!"],
    )

    docs = []
    for _, row in long_df.iterrows():
        pieces = splitter.split_text(row["retrieval_text"])
        for piece in pieces:
            docs.append(Document(
                page_content="passage: " + piece,
                metadata={"drug_name": row["drug_name"]},
            ))

    return FAISS.from_documents(docs, embedding)


def evaluate_config(vectorstore: FAISS, test_set: list[dict]) -> float:
    hits = 0
    for item in test_set:
        query = "query: " + item["query"]
        results = vectorstore.similarity_search(query, k=TOP_K)
        retrieved_drugs = {doc.metadata.get("drug_name") for doc in results}
        if item["expected_drug"] in retrieved_drugs:
            hits += 1
    return hits / len(test_set)


def main():
    df = pd.read_csv(CLEANED_PATH)
    long_df = df[df["retrieval_text"].str.len() >= LONG_RECORD_THRESHOLD].copy()
    print(f"Long records used for this comparison: {len(long_df)}")

    test_set = build_test_set(df)
    print(f"Test questions built: {len(test_set)}")

    embedding = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        encode_kwargs={"normalize_embeddings": True},
    )

    print("\n=== Results (hit rate: correct drug found in top-5) ===")
    for config in CONFIGS:
        vectorstore = build_vectorstore_for_config(
            long_df, embedding, config["chunk_size"], config["chunk_overlap"]
        )
        accuracy = evaluate_config(vectorstore, test_set)
        print(f"{config['name']}: {accuracy:.1%}")


if __name__ == "__main__":
    main()

C:\Users\Saraye Tel\AppData\Local\Temp\ipykernel_24716\3257388406.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Long records used for this comparison: 65
Test questions built: 64


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


=== Results (hit rate: correct drug found in top-5) ===
1000/200 (original): 62.5%
400/60 (proposed middle ground): 65.6%
250/30 (your latest change): 71.9%


# retrievel

In [6]:
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


PROJECT_ROOT = Path.cwd().parent
FAISS_DIR = PROJECT_ROOT / "data" / "processed" / "faiss_index"

# Must match the exact model used in build_index.py - mixing embedding
# models between indexing and querying produces meaningless results.
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

# Number of chunks to retrieve per query (k=5 per the project proposal)
TOP_K = 5

def load_vectorstore() -> FAISS:
    if not FAISS_DIR.exists():
        raise FileNotFoundError(
            f"FAISS index not found at: {FAISS_DIR}\n"
            f"Run build_index.py first to generate it."
        )

    embedding = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        encode_kwargs={"normalize_embeddings": True},
    )

    return FAISS.load_local(
        str(FAISS_DIR),
        embedding,
        allow_dangerous_deserialization=True,
    )


def search(vectorstore: FAISS, query: str, k: int = TOP_K):
    """Return the top-k most relevant chunks for a given query.

    Note: multilingual-e5 models require a "query: " prefix on search
    queries (documents were indexed with a "passage: " prefix in
    build_index.py). This is a requirement of the model itself.
    """
    prefixed_query = "query: " + query
    results = vectorstore.similarity_search_with_score(prefixed_query, k=k)
    return results


def print_results(query: str, results) -> None:
    print(f"\nQuery: {query}")
    print(f"Top {len(results)} results:\n")
    for rank, (doc, score) in enumerate(results, start=1):
        print(f"[{rank}] score={score:.4f} | drug={doc.metadata.get('drug_name')}")
        print(f"    {doc.page_content}")
        print()

vectorstore = load_vectorstore()

C:\Users\Saraye Tel\AppData\Local\Temp\ipykernel_4732\528627244.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [12]:
sample_queries = [
    "آسپرین برای سردرد خوبه؟",
    "مصرف قرص فلوکستین چه عوارضی داره؟",
    "آیا شربت لاکتولوز برای یبوست کودکان مناسب است؟",
]
for q in sample_queries:
    results = search(vectorstore, q)
    print_results(q, results)


Query: آسپرین برای سردرد خوبه؟
Top 5 results:

[1] score=0.2884 | drug=Acetaminophen-Ibuprofen-Caffeine
    passage: دارو: Acetaminophen-Ibuprofen-Caffeine | سؤال: سلام. چندروزی است سردرد شدیدی می‌گیرم طوری که فشار زیادی روی چشم چپم میاره. چند روز بیمارستان بستری بودم تمام آزمایشها ام. آر. ای. سیتیزن اسکن گرفتن هیچی متوجه نشدند. الان دکتر دیگه رفتم فارژزیک تجویز کرده آیا اثری داره …. لطفا راهنماییم کنید | پاسخ: این دارو مسکن است و خاصیت ضد درد نسبتا قوی دارد ولی اگر بیماری زمینه‌ای مانند میگرن داشته باشید درمان‌کننده علت نخواهد بود. اگر نتایج آزمایشات و سی تی اسکن مشکل خاصی نشان نداده به نظر نمی‌رسد مشکل خاصی وجود داشته باشد.

[2] score=0.2895 | drug=Sumatriptan-Naproxen
    passage: دارو: Sumatriptan-Naproxen | سؤال: سلام وقت شما بخیر چند سالی هست سر درد دارم و اوایل از داروهای مقل نوافن استفاده می‌کردم بعد متوجه شدم میگرن هست و با مراجعه به دکتر مغز و اعصاب از دپاکین استفاده می‌کردم که باعث می‌شد سردردم شدید‌تر بشه به دکتر مراجعه کردم داروها رو قطع کردم و از دارو ریزتریپتان استفاده 

In [8]:
sample_queries = [
    "دوز مصرف سفیکسیم برای کودکان چقدره؟",
    "آیا مصرف ایبوپروفن در دوران بارداری خطرناکه؟",
    "قرص فاموتیدین با چه داروهایی تداخل داره؟",
    "سیپروفلوکساسین برای عفونت ادراری چند روز باید مصرف بشه؟",
    "تفاوت فلوکستین و فلووکسامین چیه؟",  # تست تشخیص دو دارو شبیه‌هم
    "آیا موپیروسین برای زخم صورت هم استفاده میشه؟",
    "کلردیازپوکساید چه عوارض جانبی‌ای داره؟",
    "آیا میشه تئوفیلین رو با قهوه مصرف کرد؟",
]

for q in sample_queries:
    results = search(vectorstore, q)
    print_results(q, results)


Query: دوز مصرف سفیکسیم برای کودکان چقدره؟
Top 5 results:

[1] score=0.2567 | drug=Cefixime
    passage: دارو: Cefixime | سؤال: سلام، وقت بخیر. کودک ۶ ساله ۳۰ کیلویی به تجویز متخصص کودکان برای درمان سرفه خلطی و تب، یک شیشه شربت سفیکسیم را به صورت هر ۱۲ ساعت ۶ سی‌سی استفاده کرده است. اما هنوز ۷ روز نشده دارو تمام‌شده. آیا نیازی به تهیه شربت دیگر و ادامه درمان تا ۷ یا ۱۰ روز هست؟ چند روز؟ قبلا از راهنمایی شما متشکرم. | پاسخ: اگر نیاز به مقدار بیشتر آنتی بیوتیک وجود داشت پزشک تجویز کرده بود. نیازی به تهیه سفکیسم اضافه نیست.

[2] score=0.2653 | drug=Cefixime
    passage: دارو: Cefixime | سؤال: با سلام.. دکتر برای من سفکسیم ۴۰۰ تجویز کردن اما دستور مصرفش نوشته روزی دوبار.. من فکر می‌کنم این دوز خیلی بالاس بخاطر همین هنوز مصرف نکردم.. قبلا از ایتروکونازول ۱۰۰ استفاده کردم و دچار تنگی نفس شدم …لطفا راهنماییم کنید. مچکرم | پاسخ: دوز داروها توسط پزشک تجویز می‌شود که بستگی به شدت و نوع بیماری، وزن بیمار و سایر عوامل زمینه‌ای است. معمولا دوز سفکیسم برای بزرگسالان روزانه ۴۰۰ میلی‌گرم است ولی به ن

# generation

In [9]:
from pathlib import Path
import ollama

GENERATION_MODEL = "qwen2.5:3b-instruct"

SYSTEM_PROMPT = """تو یک دستیار اطلاعات دارویی هستی.

فقط بر اساس اطلاعات موجود در بخش «زمینه» (Context) پاسخ بده.
اگر پاسخ سؤال در زمینه وجود ندارد، اطلاعات را حدس نزن و صریحاً اعلام کن
که اطلاعات کافی در منابع موجود نیست.

پاسخ را به زبان فارسی، واضح و مختصر ارائه کن.

در مورد تشخیص قطعی بیماری یا تغییر خودسرانه‌ی دوز دارو، توصیه‌ی قطعی
ارائه نکن و کاربر را به مراجعه به پزشک یا داروساز ارجاع بده."""


def build_context(results) -> str:
    """Format retrieved (doc, score) pairs into a single context block."""
    parts = []
    for i, (doc, _score) in enumerate(results, start=1):
        drug = doc.metadata.get("drug_name", "نامشخص")
        # Strip the "passage: " prefix added during indexing - it's an
        # embedding-model requirement, not something the LLM should see.
        text = doc.page_content.removeprefix("passage: ")
        parts.append(f"[منبع {i} - دارو: {drug}]\n{text}")
    return "\n\n".join(parts)


def generate_answer(question: str, results) -> str:
    """Generate a Persian answer from the user's question and retrieved chunks.

    Requires the Ollama app to be running in the background (it starts
    automatically after installation on most systems).
    """
    context = build_context(results)
    prompt = f"زمینه (Context):\n{context}\n\nسؤال کاربر:\n{question}"

    response = ollama.chat(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0.2},  # low temperature: grounded, consistent answers
    )

    return response["message"]["content"]


In [10]:
question = "ازیترومایسین کاربردش چیه؟"
vectorstore = load_vectorstore()
results = search(vectorstore, question)
answer = generate_answer(question, results)

print(answer)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

ازیترومایسین برای علاوهای مختلفی مانند بیماری‌های ریهی، سینوسی، و سیستم خونی استفاده می‌شود. در برخی موارد، می‌تواند برای کاهش ضربان قلبی نیز استفاده می‌شود.


# rag_test

In [11]:

from pathlib import Path
import sys

# Allow importing retrieve.py and generator.py from the retrieval/
# and generation/ folders without turning the whole project into a
# formal installable package - fine for a small team project.
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src" / "retrieval"))
sys.path.append(str(PROJECT_ROOT / "src" / "generation"))


def answer_question(vectorstore, question: str) -> None:
    results = search(vectorstore, question)
    answer = generate_answer(question, results)

    print(f"\nسؤال: {question}")
    print(f"پاسخ: {answer}\n")
    print("منابع استفاده‌شده:")
    for i, (doc, score) in enumerate(results, start=1):
        print(f"  [{i}] {doc.metadata.get('drug_name')} (score={score:.4f})")
    print("-" * 60)


if __name__ == "__main__":
    vectorstore = load_vectorstore()

    test_questions = [
        "آسپرین برای سردرد خوبه؟",
        "تفاوت فلوکستین و فلووکسامین چیه؟",
        "دوز مصرف سفیکسیم برای کودکان چقدره؟",
    ]

    for q in test_questions:
        answer_question(vectorstore, q)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


سؤال: آسپرین برای سردرد خوبه؟
پاسخ: آسپرین برای سردرد خوبه می‌تواند باشد ولی باید با پزشک مراجعه کنید تا بتوانید بهترین درمان برای شما پیشنهاد شود.

منابع استفاده‌شده:
  [1] Acetaminophen-Ibuprofen-Caffeine (score=0.2884)
  [2] Sumatriptan-Naproxen (score=0.2895)
  [3] Sodium-Valproate (score=0.2898)
  [4] Enoxaparin-Sodium (score=0.2904)
  [5] Aspirin (score=0.2909)
------------------------------------------------------------

سؤال: تفاوت فلوکستین و فلووکسامین چیه؟
پاسخ: فلوکستین و فلووکسامین داروهای ضد افسردگی هستند و از نظر عملکرد هر دو مهارکننده انتخابی سروتونین هستند. ولی فلووکسامین معمولاً کاهش میل جنسی ایجاد می‌کند.

منابع استفاده‌شده:
  [1] Fluvoxamine (score=0.2285)
  [2] Fluvoxamine (score=0.3055)
  [3] Bupropion (score=0.3056)
  [4] Fluoxetine (score=0.3102)
  [5] Fluvoxamine (score=0.3145)
------------------------------------------------------------

سؤال: دوز مصرف سفیکسیم برای کودکان چقدره؟
پاسخ: دوز مصرف سفکسیم برای کودکان توسط پزشک تجویز می‌شود. معمولاً سفکسیم برای بزرگ

# Evaluation

In [1]:
"""
Formal evaluation against PersianMedQA - checks whether the project's
stated goal (retrieval accuracy >= 65%, Hit Rate@5) is actually met.

Methodology:
PersianMedQA has no drug_name ground truth field (unlike our own
dataset), so we can't directly reuse the earlier chunk_size comparison
script as-is. However, many "Pharmacology" field questions are
multiple-choice among DRUG NAMES themselves (e.g. "which drug is used
for X?" with drug names as the 4 options). So:

  1. We mine Persian drug-name spellings from our OWN dataset (same
     technique used earlier for the MeDiaPQA enrichment attempt): for
     each drug_name, find the Persian word that appears in the highest
     fraction of that drug's comments/responses, while excluding words
     that are too generic (appear across many different drugs).
  2. We filter PersianMedQA to field == "فارماکولوژی" (Pharmacology).
  3. For each question, we check whether the CORRECT answer option text
     matches one of our mined Persian drug names. Only matched
     questions are usable for this evaluation - a question whose
     correct answer isn't a drug our own dataset covers can't fairly
     test our retrieval system.
  4. For each usable question, we run full retrieval (k=5, same
     pipeline as production) using the question text as the query, and
     check Hit Rate@5: is the correct drug_name present among the
     top-5 retrieved chunks' metadata?

Requires: pip install pandas
          (plus everything retrieve.py already needs)
"""
import ast
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_dataset.csv"

# Adjust this path to wherever your PersianMedQA CSV actually lives.
PERSIANMEDQA_PATH = PROJECT_ROOT / "data" / "raw" / "PersianMedQA-train.csv"

TOP_K = 5
PHARMACOLOGY_FIELD = "فارماکولوژی"

# Same thresholds as the mining step used for the MeDiaPQA experiment.
STOPWORD_DOC_FREQ_THRESHOLD = 0.25
MIN_COVERAGE = 0.30

ARABIC_TO_PERSIAN = {"ي": "ی", "ك": "ک", "ة": "ه", "ۀ": "ه", "أ": "ا", "إ": "ا", "ؤ": "و", "ئ": "ی"}


def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("_x000D_", " ").replace("\r", " ").replace("\n", " ")
    for ar, fa in ARABIC_TO_PERSIAN.items():
        text = text.replace(ar, fa)
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text: str) -> set:
    return set(re.findall(r"[\u0600-\u06FF]{3,}", text))


def mine_persian_drug_names(cleaned_df: pd.DataFrame) -> dict:
    """Return {persian_name: english_drug_name} mined from our own dataset."""
    group_doc_freq = Counter()
    row_presence = defaultdict(lambda: defaultdict(int))
    group_sizes = {}

    combined_text = cleaned_df["comment_clean"].fillna("") + " " + cleaned_df["response_clean"].fillna("")

    for drug, group in cleaned_df.assign(_text=combined_text).groupby("drug_name"):
        group_sizes[drug] = len(group)
        words_in_group = set()
        for t in group["_text"]:
            toks = tokenize(t)
            words_in_group |= toks
            for w in toks:
                row_presence[drug][w] += 1
        for w in words_in_group:
            group_doc_freq[w] += 1

    total_groups = len(group_sizes)
    candidates = {}
    for drug in group_sizes:
        best_word, best_score = None, 0
        for w, cnt in row_presence[drug].items():
            if group_doc_freq[w] / total_groups > STOPWORD_DOC_FREQ_THRESHOLD:
                continue
            coverage = cnt / group_sizes[drug]
            if coverage > best_score:
                best_score, best_word = coverage, w
        if best_word and best_score >= MIN_COVERAGE:
            candidates[best_word] = drug

    return candidates


def build_test_set(persianmedqa_df: pd.DataFrame, persian_to_english: dict) -> list[dict]:
    """Match PersianMedQA pharmacology questions to a known drug_name."""
    pharma_df = persianmedqa_df[persianmedqa_df["field"] == PHARMACOLOGY_FIELD]

    test_set = []
    for _, row in pharma_df.iterrows():
        try:
            options = ast.literal_eval(row["answer"])
            correct_text = options[str(row["correct answer"])].strip()
        except (ValueError, KeyError, SyntaxError):
            continue

        normalized_correct = normalize(correct_text)
        expected_drug = None
        for persian_name, english_name in persian_to_english.items():
            if persian_name in normalized_correct or normalized_correct in persian_name:
                expected_drug = english_name
                break

        if expected_drug:
            test_set.append({"query": row["question"], "expected_drug": expected_drug})

    return test_set


def evaluate(vectorstore, test_set: list[dict], show_failures: int = 0) -> float:
    hits = 0
    failures_shown = 0
    for item in test_set:
        query = "query: " + item["query"]
        results = vectorstore.similarity_search(query, k=TOP_K)
        retrieved_drugs = {doc.metadata.get("drug_name") for doc in results}
        if item["expected_drug"] in retrieved_drugs:
            hits += 1
        elif failures_shown < show_failures:
            # Print a few misses so we can inspect WHY retrieval failed -
            # e.g. domain/register mismatch vs a genuinely wrong match.
            print(f"\n[MISS] question: {item['query'][:120]}")
            print(f"       expected_drug: {item['expected_drug']}")
            print(f"       retrieved_drugs: {retrieved_drugs}")
            failures_shown += 1
    return hits / len(test_set) if test_set else 0.0


def main():
    if not CLEANED_PATH.exists():
        raise FileNotFoundError(f"Cleaned dataset not found at: {CLEANED_PATH}")
    if not PERSIANMEDQA_PATH.exists():
        raise FileNotFoundError(
            f"PersianMedQA CSV not found at: {PERSIANMEDQA_PATH}\n"
            f"Edit PERSIANMEDQA_PATH at the top of this script if it's stored elsewhere."
        )

    from retrieval.retrieve import load_vectorstore  # local import: needs sys.path set by caller

    cleaned_df = pd.read_csv(CLEANED_PATH)
    persianmedqa_df = pd.read_csv(PERSIANMEDQA_PATH)

    print("Mining Persian drug-name spellings from our own dataset...")
    persian_to_english = mine_persian_drug_names(cleaned_df)
    print(f"Mined {len(persian_to_english)} reliable drug-name candidates")

    test_set = build_test_set(persianmedqa_df, persian_to_english)
    print(f"Usable PersianMedQA pharmacology questions (answer matches a known drug): {len(test_set)}")

    if not test_set:
        print("No usable test questions found - check PERSIANMEDQA_PATH and field name.")
        return

    print("Loading vector store (this may take a moment)...")
    vectorstore = load_vectorstore()

    print("Running evaluation...")
    accuracy = evaluate(vectorstore, test_set, show_failures=10)

    print("\n=== Result ===")
    print(f"Hit Rate@{TOP_K}: {accuracy:.1%}  (target: >= 65%)")
    print(f"Based on {len(test_set)} PersianMedQA pharmacology questions")
    if accuracy >= 0.65:
        print("Target MET.")
    else:
        print("Target NOT met - see project report for discussion.")


if __name__ == "__main__":
    import sys
    sys.path.append(str(PROJECT_ROOT / "src"))
    main()

Mining Persian drug-name spellings from our own dataset...
Mined 333 reliable drug-name candidates
Usable PersianMedQA pharmacology questions (answer matches a known drug): 193
Loading vector store (this may take a moment)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Running evaluation...

[MISS] question: کدام دسته از داروهای زیر ممکن است باعث اختلال در رشد غضروف شود؟
       expected_drug: Primaquine
       retrieved_drugs: {'Methotrexate', 'Potasium-Citrate', 'Zoledronic-Acid', 'Methylphenidate', 'Methyltestosterone'}

[MISS] question: کدام گزینه درخصوص مقایسه دیورتیک های لوپ و تیازیدها صحیح است؟
       expected_drug: Lidocaine-Systemic
       retrieved_drugs: {'Loperamide', 'Methotrexate', 'Escitalopram', 'Levetiracetam'}

[MISS] question: بیمار فردی 40 ساله است که به علت بروز عفونت مننژیت ناشی از هموفیلوس آنفلوآنزا در بخش عفونی بستری شده است. در آزمایش خون 
       expected_drug: Trifluoperazine
       retrieved_drugs: {'Doxapram', 'Allopurinol', 'Amantadine', 'Ciprofloxacin', 'Metformin'}

[MISS] question: افزایش ترشح انسولین مکانیسم اصلی کدام داروی کاهنده قند زیر است؟
       expected_drug: Gliclazide
       retrieved_drugs: {'Repaglinide', 'Metformin', 'Gemfibrozil', 'Buprenorphine'}

[MISS] question: کدام یک از آنتی کلی نرژیک های ذیل از نیمه 

In [3]:
"""
Formal evaluation of the FINAL production system (chunk_size=250,
multilingual-e5-large, current FAISS index) against our own dataset -
the counterpart to evaluate_persianmedqa.py, representing the system's
actual intended use case (colloquial patient questions), not clinical
exam-style questions.

Methodology (same anti-leakage design as compare_chunk_sizes.py):
the test query is a short excerpt from ~60% into the RESPONSE text
(not the question), so the query is not a verbatim substring sitting
at the very start of the chunk that was embedded - this avoids a
trivially easy "find myself" retrieval task.

Unlike compare_chunk_sizes.py (which only tested the ~55 long records
and rebuilt temporary indices), this script tests a sample from the
WHOLE dataset against the actual production FAISS index already built
by build_index.py - this is the real, final system.

Requires: pip install pandas
          (plus everything retrieve.py already needs)
"""
import random
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_dataset.csv"

TOP_K = 5
MIN_RESPONSE_LEN = 60      # skip responses too short to safely excerpt from
EXCERPT_LEN = 40
EXCERPT_OFFSET_FRACTION = 0.6

# Set to None to evaluate the ENTIRE dataset (slower, thousands of
# vector searches). A sample keeps runtime reasonable while still being
# a statistically meaningful, randomly chosen cross-section of drugs.
SAMPLE_SIZE = None
RANDOM_SEED = 42


def build_test_set(df: pd.DataFrame) -> list[dict]:
    test_set = []
    for _, row in df.iterrows():
        response = row["response_clean"]
        if not isinstance(response, str) or len(response) < MIN_RESPONSE_LEN:
            continue
        start = int(len(response) * EXCERPT_OFFSET_FRACTION)
        excerpt = response[start:start + EXCERPT_LEN].strip()
        if len(excerpt) < 20:
            continue
        test_set.append({"query": excerpt, "expected_drug": row["drug_name"]})
    return test_set


def evaluate(vectorstore, test_set: list[dict], show_failures: int = 0) -> float:
    hits = 0
    failures_shown = 0
    for item in test_set:
        query = "query: " + item["query"]
        results = vectorstore.similarity_search(query, k=TOP_K)
        retrieved_drugs = {doc.metadata.get("drug_name") for doc in results}
        if item["expected_drug"] in retrieved_drugs:
            hits += 1
        elif failures_shown < show_failures:
            print(f"\n[MISS] excerpt: {item['query']}")
            print(f"       expected_drug: {item['expected_drug']}")
            print(f"       retrieved_drugs: {retrieved_drugs}")
            failures_shown += 1
    return hits / len(test_set) if test_set else 0.0


def main():
    if not CLEANED_PATH.exists():
        raise FileNotFoundError(f"Cleaned dataset not found at: {CLEANED_PATH}")

    sys.path.append(str(PROJECT_ROOT / "src"))
    from retrieval.retrieve import load_vectorstore

    df = pd.read_csv(CLEANED_PATH)
    print(f"Total records in dataset: {len(df)}")

    test_set = build_test_set(df)
    print(f"Usable test questions (long enough response): {len(test_set)}")

    if SAMPLE_SIZE and len(test_set) > SAMPLE_SIZE:
        random.seed(RANDOM_SEED)
        test_set = random.sample(test_set, SAMPLE_SIZE)
        print(f"Sampled down to {SAMPLE_SIZE} for faster evaluation")

    print("Loading production vector store (data/processed/faiss_index)...")
    vectorstore = load_vectorstore()

    print("Running evaluation...")
    accuracy = evaluate(vectorstore, test_set, show_failures=5)

    print("\n=== Result ===")
    print(f"Hit Rate@{TOP_K}: {accuracy:.1%}  (target: >= 65%)")
    print(f"Based on {len(test_set)} questions from our own dataset")
    if accuracy >= 0.65:
        print("Target MET.")
    else:
        print("Target NOT met.")


if __name__ == "__main__":
    main()


Total records in dataset: 4718
Usable test questions (long enough response): 2760
Loading production vector store (data/processed/faiss_index)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Running evaluation...

[MISS] excerpt: روزی یک عدد آسپرین مصرف کنید.
       expected_drug: Aspirin
       retrieved_drugs: {'Bisacodyl', 'Cefixime', 'Ichthyol', 'Spironolactone'}

[MISS] excerpt: ون بدهید (فصد) همین طور دلتان راخوش کنید
       expected_drug: Aspirin
       retrieved_drugs: {'Donepezil', 'Lansoprazole', 'Tamsulosin', 'Rituximab', 'Lactulose'}

[MISS] excerpt: بقه خونریزی معده ندارید مشکلی نیست
       expected_drug: Aspirin
       retrieved_drugs: {'Phenobarbital', 'Azithromycin', 'Tetanus-Vaccines', 'Famotidine', 'Nitrofurantoin'}

[MISS] excerpt: ن؟ باهم خوردین یا با فاصله؟
       expected_drug: Aspirin
       retrieved_drugs: {'Mesalazine', 'Sertraline', 'Alprazolam', 'Phenazopyridine', 'Famotidine'}

[MISS] excerpt: زانه لازم است و بهتر است تحت نظر پزشک مص
       expected_drug: Aspirin
       retrieved_drugs: {'Allopurinol', 'Silver-Sulphadiazine-Topical', 'Acetazolamide', 'Sodium-Chloride-Nasal', 'Piracetam'}

=== Result ===
Hit Rate@5: 50.0%  (target: >= 65%)
Based

In [1]:
"""
Formal evaluation of the FINAL production system (chunk_size=250,
multilingual-e5-large, current FAISS index) against our own dataset -
the counterpart to evaluate_persianmedqa.py, representing the system's
actual intended use case (colloquial patient questions), not clinical
exam-style questions.

Methodology (same anti-leakage design as compare_chunk_sizes.py):
the test query is a short excerpt from ~60% into the RESPONSE text
(not the question), so the query is not a verbatim substring sitting
at the very start of the chunk that was embedded - this avoids a
trivially easy "find myself" retrieval task.

Unlike compare_chunk_sizes.py (which only tested the ~55 long records
and rebuilt temporary indices), this script tests a sample from the
WHOLE dataset against the actual production FAISS index already built
by build_index.py - this is the real, final system.

Requires: pip install pandas
          (plus everything retrieve.py already needs)
"""
import random
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_dataset.csv"

TOP_K = 5
EXCERPT_WINDOW = 60  # characters of context kept on each side of the drug name

# Same thresholds as the drug-name mining step used for the
# MeDiaPQA/PersianMedQA experiments - a word only counts as a drug's
# Persian name if it's specific enough (not too common across other
# drugs' records) and appears reliably often within that drug's own
# records.
STOPWORD_DOC_FREQ_THRESHOLD = 0.25
MIN_COVERAGE = 0.30

# Set to None to evaluate the ENTIRE dataset (slower, thousands of
# vector searches). A sample keeps runtime reasonable while still being
# a statistically meaningful, randomly chosen cross-section of drugs.
SAMPLE_SIZE = 300
RANDOM_SEED = 42


def tokenize(text: str) -> set:
    return set(re.findall(r"[\u0600-\u06FF]{3,}", text))


def mine_persian_drug_names(df: pd.DataFrame) -> dict:
    """Return {english_drug_name: persian_name} mined from our own dataset.

    For each drug, find the Persian word that appears in the highest
    fraction of that drug's own comments/responses, while excluding
    words too generic to be specific to one drug (e.g. "دارو", "مصرف").
    """
    combined_text = df["comment_clean"].fillna("") + " " + df["response_clean"].fillna("")
    group_doc_freq = Counter()
    row_presence = defaultdict(lambda: defaultdict(int))
    group_sizes = {}

    for drug, group in df.assign(_text=combined_text).groupby("drug_name"):
        group_sizes[drug] = len(group)
        words_in_group = set()
        for t in group["_text"]:
            toks = tokenize(t)
            words_in_group |= toks
            for w in toks:
                row_presence[drug][w] += 1
        for w in words_in_group:
            group_doc_freq[w] += 1

    total_groups = len(group_sizes)
    names = {}
    for drug in group_sizes:
        best_word, best_score = None, 0
        for w, cnt in row_presence[drug].items():
            if group_doc_freq[w] / total_groups > STOPWORD_DOC_FREQ_THRESHOLD:
                continue
            coverage = cnt / group_sizes[drug]
            if coverage > best_score:
                best_score, best_word = coverage, w
        if best_word and best_score >= MIN_COVERAGE:
            names[drug] = best_word
    return names


def build_test_set(df: pd.DataFrame, drug_persian_names: dict) -> list[dict]:
    """Build test queries centered on an actual mention of the drug's
    own Persian name within that row's own text - not an arbitrary
    offset. Rows where the drug's mined name doesn't literally appear
    are skipped, since we can't build a fair, specific query for them
    this way."""
    test_set = []
    for _, row in df.iterrows():
        persian_name = drug_persian_names.get(row["drug_name"])
        if not persian_name:
            continue

        text = row["response_clean"]
        if not isinstance(text, str):
            continue

        idx = text.find(persian_name)
        if idx == -1:
            continue

        start = max(0, idx - EXCERPT_WINDOW // 2)
        end = min(len(text), idx + len(persian_name) + EXCERPT_WINDOW // 2)
        excerpt = text[start:end].strip()

        test_set.append({"query": excerpt, "expected_drug": row["drug_name"]})
    return test_set


def evaluate(vectorstore, test_set: list[dict], show_failures: int = 0) -> float:
    hits = 0
    failures_shown = 0
    for item in test_set:
        query = "query: " + item["query"]
        results = vectorstore.similarity_search(query, k=TOP_K)
        retrieved_drugs = {doc.metadata.get("drug_name") for doc in results}
        if item["expected_drug"] in retrieved_drugs:
            hits += 1
        elif failures_shown < show_failures:
            print(f"\n[MISS] excerpt: {item['query']}")
            print(f"       expected_drug: {item['expected_drug']}")
            print(f"       retrieved_drugs: {retrieved_drugs}")
            failures_shown += 1
    return hits / len(test_set) if test_set else 0.0


def main():
    if not CLEANED_PATH.exists():
        raise FileNotFoundError(f"Cleaned dataset not found at: {CLEANED_PATH}")

    sys.path.append(str(PROJECT_ROOT / "src"))
    from retrieval.retrieve import load_vectorstore

    df = pd.read_csv(CLEANED_PATH)
    print(f"Total records in dataset: {len(df)}")

    print("Mining Persian drug-name spellings from our own dataset...")
    drug_persian_names = mine_persian_drug_names(df)
    print(f"Mined {len(drug_persian_names)} reliable drug-name candidates")

    test_set = build_test_set(df, drug_persian_names)
    print(f"Usable test questions (row's own text contains its drug's mined name): {len(test_set)}")

    if SAMPLE_SIZE and len(test_set) > SAMPLE_SIZE:
        random.seed(RANDOM_SEED)
        test_set = random.sample(test_set, SAMPLE_SIZE)
        print(f"Sampled down to {SAMPLE_SIZE} for faster evaluation")

    print("Loading production vector store (data/processed/faiss_index)...")
    vectorstore = load_vectorstore()

    print("Running evaluation...")
    accuracy = evaluate(vectorstore, test_set, show_failures=5)

    print("\n=== Result ===")
    print(f"Hit Rate@{TOP_K}: {accuracy:.1%}  (target: >= 65%)")
    print(f"Based on {len(test_set)} questions from our own dataset")
    if accuracy >= 0.65:
        print("Target MET.")
    else:
        print("Target NOT met.")


if __name__ == "__main__":
    main()

Total records in dataset: 4718
Mining Persian drug-name spellings from our own dataset...
Mined 414 reliable drug-name candidates
Usable test questions (row's own text contains its drug's mined name): 1083
Sampled down to 300 for faster evaluation
Loading production vector store (data/processed/faiss_index)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Running evaluation...

[MISS] excerpt: سلام لطفا به هیچ عنوان از رانیتیدین در حال حاضر استفاده نکنید. و
       expected_drug: Ranitidine
       retrieved_drugs: {'Alprazolam', 'Nortriptyline', 'Cyproterone-Acetate', 'Methyltestosterone', 'Tetracosactide'}

[MISS] excerpt: بله. احتمال زیاد عوارض والزارتان است.
       expected_drug: Valsartan
       retrieved_drugs: {'Trifluoperazine', 'Amantadine', 'Contraceptive-DE', 'Clindamycin', 'Lithium-Carbonate'}

[MISS] excerpt: سلام، خیر نباید استفاده کنن.
       expected_drug: Buprenorphine-Naloxone
       retrieved_drugs: {'Nortriptyline', 'Cyproterone-Acetate', 'Phenazopyridine', 'Mupirocin', 'Spironolactone'}

[MISS] excerpt: یاد این قطره تحریک‌کننده مخاط بینی هست
       expected_drug: Sodium-Chloride-Nasal
       retrieved_drugs: {'Cyproterone-Acetate', 'Rosuvastatin', 'Topiramate', 'Mupirocin', 'Acetazolamide'}

[MISS] excerpt: سلام خیر اصلا برای افزایش از این دارو استفاده نکنید حتم
       expected_drug: Megestrol
       retrieved_drugs: 

## Benchmark response time

In [3]:
"""
Measures end-to-end response time of the live API (retrieval +
generation, exactly as a real user would experience it) against the
project proposal's target of under 5 seconds per answer.

This calls the actual running FastAPI server over HTTP, not the
internal Python functions directly - so the measured time includes
everything a real request goes through (network, FastAPI routing,
embedding the query, FAISS search, and the Ollama generation call).

Run: python benchmark_response_time.py
Requires: the FastAPI server must already be running (uvicorn main:app)

Requires: pip install requests
"""
import statistics
import time

import requests

API_URL = "http://127.0.0.1:8000/ask"  # change if the server runs elsewhere

# A realistic mix of questions - not cherry-picked "easy" ones - so the
# measured time reflects normal usage, not a best case.
TEST_QUESTIONS = [
    "آسپرین برای سردرد خوبه؟",
    "دوز مصرف سفیکسیم برای کودکان چقدره؟",
    "تفاوت فلوکستین و فلووکسامین چیه؟",
    "قرص فاموتیدین با چه داروهایی تداخل داره؟",
    "آیا شربت لاکتولوز برای یبوست کودکان مناسب است؟",
    "کلردیازپوکساید چه عوارض جانبی‌ای داره؟",
    "سیپروفلوکساسین برای عفونت ادراری چند روز باید مصرف بشه؟",
    "آیا موپیروسین برای زخم صورت هم استفاده میشه؟",
    "آیا میشه تئوفیلین رو با قهوه مصرف کرد؟",
    "نظرت درباره داروهای گیاهی چینی چیه؟",  # deliberately out-of-domain, tests the "no confident match" path too
]

TARGET_SECONDS = 5.0


def measure_one(question: str) -> float:
    start = time.perf_counter()
    response = requests.post(API_URL, json={"question": question}, timeout=120)
    elapsed = time.perf_counter() - start
    response.raise_for_status()
    return elapsed


def main():
    print(f"Sending {len(TEST_QUESTIONS)} questions to {API_URL}...\n")
    times = []

    for i, question in enumerate(TEST_QUESTIONS, start=1):
        try:
            elapsed = measure_one(question)
        except requests.RequestException as e:
            print(f"[{i}] FAILED - {e}")
            continue

        times.append(elapsed)
        flag = "OK" if elapsed <= TARGET_SECONDS else "SLOW"
        print(f"[{i}] {elapsed:5.2f}s  [{flag}]  {question[:50]}")

    if not times:
        print("\nNo successful requests - is the server running?")
        return

    print("\n=== Result ===")
    print(f"Average: {statistics.mean(times):.2f}s")
    print(f"Median:  {statistics.median(times):.2f}s")
    print(f"Min:     {min(times):.2f}s")
    print(f"Max:     {max(times):.2f}s")
    within_target = sum(1 for t in times if t <= TARGET_SECONDS)
    print(f"\n{within_target}/{len(times)} requests were within the {TARGET_SECONDS:.0f}s target.")


if __name__ == "__main__":
    main()

Sending 10 questions to http://127.0.0.1:8000/ask...

[1] 40.49s  [SLOW]  آسپرین برای سردرد خوبه؟
[2] 18.03s  [SLOW]  دوز مصرف سفیکسیم برای کودکان چقدره؟
[3] 16.56s  [SLOW]  تفاوت فلوکستین و فلووکسامین چیه؟
[4] 22.25s  [SLOW]  قرص فاموتیدین با چه داروهایی تداخل داره؟
[5] 20.84s  [SLOW]  آیا شربت لاکتولوز برای یبوست کودکان مناسب است؟
[6] 12.98s  [SLOW]  کلردیازپوکساید چه عوارض جانبی‌ای داره؟
[7]  9.03s  [SLOW]  سیپروفلوکساسین برای عفونت ادراری چند روز باید مصرف
[8] 18.40s  [SLOW]  آیا موپیروسین برای زخم صورت هم استفاده میشه؟
[9]  6.71s  [SLOW]  آیا میشه تئوفیلین رو با قهوه مصرف کرد؟
[10] 16.47s  [SLOW]  نظرت درباره داروهای گیاهی چینی چیه؟

=== Result ===
Average: 18.18s
Median:  17.29s
Min:     6.71s
Max:     40.49s

0/10 requests were within the 5s target.


## compare_abstention_methods

In [1]:
"""
Adapted from the instructor's compare_eval.py template (fine-tuned
model vs prompt-engineering comparison) - repurposed for OUR project's
actual need: filling the "Abstention Accuracy: not formally measured"
gap identified in the evaluation report.

We have no fine-tuned classifier (out of scope for this project), so
instead we compare our two REAL candidate methods for deciding whether
a question is in-domain (answerable from our pharmaceutical dataset)
or out-of-domain (should trigger Abstention):

  Method A - "Threshold-based" (what production actually uses):
      run real retrieval and check the best FAISS similarity score
      against MIN_CONFIDENCE_SCORE.

  Method B - "Prompt-based": ask the LLM directly, in a single
      yes/no-style prompt, whether the question is about medication.

Requires: pip install ollama
          (plus everything retrieve.py already needs)
"""
import sys
import time
from pathlib import Path

import ollama

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from retrieval.retrieve import load_vectorstore, search  # noqa: E402

GENERATION_MODEL = "qwen2.5:7b-instruct"
MIN_CONFIDENCE_SCORE = 0.30  # keep in sync with src/api/main.py

# A small labeled test set: real in-domain drug questions used
# throughout this project's manual testing, plus clearly out-of-domain
# questions used earlier to test the Abstention path.
TEST_SAMPLES = [
    {"text": "آسپرین برای سردرد خوبه؟", "true_label": "in_domain"},
    {"text": "دوز مصرف سفیکسیم برای کودکان چقدره؟", "true_label": "in_domain"},
    {"text": "تفاوت فلوکستین و فلووکسامین چیه؟", "true_label": "in_domain"},
    {"text": "قرص فاموتیدین با چه داروهایی تداخل داره؟", "true_label": "in_domain"},
    {"text": "آیا شربت لاکتولوز برای یبوست کودکان مناسب است؟", "true_label": "in_domain"},
    {"text": "کلردیازپوکساید چه عوارض جانبی‌ای داره؟", "true_label": "in_domain"},
    {"text": "سیپروفلوکساسین برای عفونت ادراری چند روز باید مصرف بشه؟", "true_label": "in_domain"},
    {"text": "آیا موپیروسین برای زخم صورت هم استفاده میشه؟", "true_label": "in_domain"},
    {"text": "نظرت درباره داروهای گیاهی چینی چیه؟", "true_label": "out_of_domain"},
    {"text": "بهترین رژیم غذایی برای کاهش وزن چیه؟", "true_label": "out_of_domain"},
    {"text": "پایتخت فرانسه کجاست؟", "true_label": "out_of_domain"},
    {"text": "بهترین گوشی موبایل زیر ده میلیون تومان کدومه؟", "true_label": "out_of_domain"},
    {"text": "نظرت درباره وضعیت اقتصاد ایران چیه؟", "true_label": "out_of_domain"},
]


def evaluate_threshold_method(samples, vectorstore):
    print("Evaluating Method A: embedding confidence threshold...")
    results = []
    start = time.time()

    for sample in samples:
        query = "query: " + sample["text"]
        search_results = vectorstore.similarity_search_with_score(query, k=5)
        best_score = min(score for _, score in search_results) if search_results else float("inf")
        predicted = "in_domain" if best_score <= MIN_CONFIDENCE_SCORE else "out_of_domain"
        results.append({
            "text": sample["text"],
            "true": sample["true_label"],
            "predicted": predicted,
            "correct": predicted == sample["true_label"],
        })

    elapsed = time.time() - start
    return results, elapsed


def evaluate_prompt_method(samples):
    print(f"Evaluating Method B: prompt-based classification with {GENERATION_MODEL}...")
    results = []
    start = time.time()

    for sample in samples:
        prompt = (
            "آیا سؤال زیر دربارهٔ یک داروی خاص (مصرف، دوز، عوارض یا تداخل دارویی) است؟\n\n"
            f"سؤال: {sample['text']}\n\n"
            "فقط یکی از این دو کلمه را بنویس: بله یا خیر"
        )
        response = ollama.chat(
            model=GENERATION_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0},
        )
        answer_text = response["message"]["content"].strip()
        predicted = "in_domain" if "بله" in answer_text else "out_of_domain"
        results.append({
            "text": sample["text"],
            "true": sample["true_label"],
            "predicted": predicted,
            "correct": predicted == sample["true_label"],
        })

    elapsed = time.time() - start
    return results, elapsed


def print_comparison(a_results, a_time, b_results, b_time):
    print("\n" + "=" * 90)
    print(f"{'متن':<45} {'واقعی':<14} {'آستانه':<14} {'پرامپت':<14}")
    print("=" * 90)

    for a, b in zip(a_results, b_results):
        text = a["text"][:42]
        true = a["true"]
        a_pred = f"{'OK' if a['correct'] else 'ERR'} {a['predicted']}"
        b_pred = f"{'OK' if b['correct'] else 'ERR'} {b['predicted']}"
        print(f"{text:<45} {true:<14} {a_pred:<14} {b_pred:<14}")

    a_acc = sum(1 for r in a_results if r["correct"]) / len(a_results)
    b_acc = sum(1 for r in b_results if r["correct"]) / len(b_results)

    print("\n" + "=" * 90)
    print("مقایسه نهایی (Abstention Accuracy):")
    print("=" * 90)
    print("روش آستانهٔ Embedding (تولید فعلی):")
    print(f"  دقت: {a_acc:.1%}")
    print(f"  زمان کل: {a_time:.2f} ثانیه ({a_time/len(a_results):.2f}s / سؤال)")
    print()
    print(f"روش Prompt-based ({GENERATION_MODEL}):")
    print(f"  دقت: {b_acc:.1%}")
    print(f"  زمان کل: {b_time:.2f} ثانیه ({b_time/len(b_results):.2f}s / سؤال)")


def main():
    print("مقایسه دو روش تشخیص سؤال دارویی/غیردارویی (Abstention)")
    print("=" * 90)

    vectorstore = load_vectorstore()

    a_results, a_time = evaluate_threshold_method(TEST_SAMPLES, vectorstore)
    b_results, b_time = evaluate_prompt_method(TEST_SAMPLES)

    print_comparison(a_results, a_time, b_results, b_time)


if __name__ == "__main__":
    main()


مقایسه دو روش تشخیص سؤال دارویی/غیردارویی (Abstention)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Evaluating Method A: embedding confidence threshold...
Evaluating Method B: prompt-based classification with qwen2.5:7b-instruct...

متن                                           واقعی          آستانه         پرامپت        
آسپرین برای سردرد خوبه؟                       in_domain      OK in_domain   OK in_domain  
دوز مصرف سفیکسیم برای کودکان چقدره؟           in_domain      OK in_domain   OK in_domain  
تفاوت فلوکستین و فلووکسامین چیه؟              in_domain      OK in_domain   OK in_domain  
قرص فاموتیدین با چه داروهایی تداخل داره؟      in_domain      OK in_domain   OK in_domain  
آیا شربت لاکتولوز برای یبوست کودکان مناسب     in_domain      OK in_domain   ERR out_of_domain
کلردیازپوکساید چه عوارض جانبی‌ای داره؟        in_domain      OK in_domain   OK in_domain  
سیپروفلوکساسین برای عفونت ادراری چند روز ب    in_domain      OK in_domain   OK in_domain  
آیا موپیروسین برای زخم صورت هم استفاده میش    in_domain      OK in_domain   OK in_domain  
نظرت درباره داروهای گیاهی چینی چیه؟          